# Response Time Model

**Goal:** training a regression model on ASSISTments 2009-2010 to predict `ms_first_response`,  
then apply it to our real dataset to fill the missing column.

**Pipeline:**
1. Load & clean ASSISTments 2009-2010
2. Train / test split 
3. Engineer features 
4. Compare Random Forest vs Gradient Boosting
5. Evaluate best model on held-out test set
6. Apply model to our dataset & save `interactions_clean.csv`

---
> **ASSISTments download:**  
> https://sites.google.com/site/assistmentsdata/home/2009-2010-assistment-data/skill-builder-data-2009-2010  
> File: `skill_builder_data_corrected_collapsed.csv`


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## Configuration

In [2]:
ASSISTMENTS_PATH  = "../Dataset/raw/skill_builder_data_corrected_collapsed.csv"
REAL_DATASET_PATH = "../Dataset/raw/interactions_raw.csv"
OUTPUT_PATH       = "../Dataset/clean/interactions_clean.csv"
MODEL_SAVE_PATH   = "../models/response_time_model.pkl"

#  Difficulty thresholds
EASY_THRESHOLD = 0.65    # correct_rate >= this  â†’  easy
HARD_THRESHOLD = 0.35    # correct_rate <= this  â†’  hard

#Response time outlier bounds 
MS_CAP   = 300_000   # 5 minutes â€” removes "walked away" responses
MS_FLOOR = 1_000     # 1 second  â€” minimum response

# Features used for training 
# We only use the common columns between both datasets
FEATURES = [
    "correct",                # 0 or 1
    "difficulty_encoded",     # 0 = easy  |  1 = medium  |  2 = hard
    "student_correct_rate",   # student's overall correct rate in the dataset
    "question_correct_rate",  # question's correct rate across all students
]


## Helper Functions

In [3]:
def assign_difficulty(rate):
    # map a question's correct rate to easy / medium / hard
    if rate >= EASY_THRESHOLD:
        return "easy"
    elif rate <= HARD_THRESHOLD:
        return "hard"
    else:
        return "medium"


def compute_rates_from_train(train_df):
    
    # Compute per-question and per-student correct rates FROM TRAINING DATA ONLY.
    # Returns two Series (indexed by problem_id and user_id) plus global fallback means.
    # These are later joined onto the test set and prediction set  never recomputed
    # from those sets  to prevent data leakage.
    
    q_rate = train_df.groupby("Question ID")["correct"].mean().rename("question_correct_rate")
    s_rate = train_df.groupby("Student ID")["correct"].mean().rename("student_correct_rate")

    global_q_mean = float(q_rate.mean())
    global_s_mean = float(s_rate.mean())

    return q_rate, s_rate, global_q_mean, global_s_mean


def apply_rates(df, q_rate, s_rate, global_q_mean, global_s_mean):
    
    # Join train-derived rates onto any DataFrame.
    # Unseen students / questions are filled with the training global mean
    # so the model always receives a valid numeric value.

    df = df.copy()
    df["question_correct_rate"] = df["Question ID"].map(q_rate).fillna(global_q_mean)
    df["student_correct_rate"]  = df["Student ID"].map(s_rate).fillna(global_s_mean)

    # Difficulty is also derived from training question rates only
    df["difficulty"]         = df["question_correct_rate"].apply(assign_difficulty)
    df["difficulty_encoded"] = df["difficulty"].map({"easy": 0, "medium": 1, "hard": 2})
    return df


def print_eval(model_name, y_true_log, y_pred_log):
    #Print MAE, RMSE, RÂ² for a model evaluated on log-scale predictions."""
    y_true_ms = np.expm1(y_true_log)
    y_pred_ms = np.expm1(y_pred_log)
    mae  = mean_absolute_error(y_true_ms, y_pred_ms)
    rmse = np.sqrt(mean_squared_error(y_true_ms, y_pred_ms))
    r2   = r2_score(y_true_log, y_pred_log)
    print(f"  [{model_name}]")
    print(f"    R\u00b2   (log scale) : {r2:.4f}")
    print(f"    MAE  (ms)        : {mae:>10,.0f}   (~{mae/1000:.1f}s average error)")
    print(f"    RMSE (ms)        : {rmse:>10,.0f}")
    return {"name": model_name, "r2": r2, "mae": mae, "rmse": rmse}


# 1. Loading & cleaning the data

In [4]:
assist = pd.read_csv(ASSISTMENTS_PATH, encoding="ISO-8859-15", low_memory=False)
print(f"Raw shape: {assist.shape}")
assist.head(3)


Raw shape: (346860, 31)


,Unnamed: 0,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0


In [5]:
#print the statistics
print(assist.describe())

          Unnamed: 0      order_id  assignment_id        user_id  \
count  346860.000000  3.468600e+05  346860.000000  346860.000000   
mean   211268.603659  3.060426e+07  273696.362801   83491.787026   
std    117373.737194  5.256154e+06   11062.932570    7328.617764   
min         1.000000  2.022408e+07  217900.000000      14.000000   
25%    108576.750000  2.652993e+07  266798.000000   78972.000000   
50%    219002.500000  3.105405e+07  271631.000000   80223.000000   
75%    312585.250000  3.485916e+07  279135.000000   88143.000000   
max    401756.000000  3.831020e+07  291503.000000   96299.000000   

       assistment_id     problem_id       original        correct  \
count  346860.000000  346860.000000  346860.000000  346860.000000   
mean    47215.410728   82903.669028       0.794147       0.645269   
std     11932.123078   25596.334242       0.404324       0.478432   
min        86.000000      83.000000       0.000000       0.000000   
25%     37546.000000   59802.000000       

In [6]:
# Check for missing values
print("Missing values in each column:")
print(assist.isnull().sum())

Missing values in each column:
Unnamed: 0                   0
order_id                     0
assignment_id                0
user_id                      0
assistment_id                0
problem_id                   0
original                     0
correct                      0
attempt_count                0
ms_first_response            0
tutor_mode                   0
answer_type                  0
sequence_id                  0
student_class_id             0
position                     0
type                         0
base_sequence_id             0
skill_id                 63755
skill_name               72270
teacher_id                   0
school_id                    0
hint_count                   0
hint_total                   0
overlap_time                 0
template_id                  0
answer_id               306818
answer_text              77630
first_action                 0
bottom_hint             287003
opportunity                  0
opportunity_original     71402
dtype: i

- Note that the missing values are in the columns that we won't need to use for the prediction since we'll keep only the common ones with our real dataset

In [7]:
# we keep only the columns we need and rename them to match our real dataset's 
# ASSISTments â†’ our schema:
#   user_id            â†’ Student ID
#   problem_id         â†’ Question ID
#   correct            â†’ correct      (same)
#   ms_first_response  â†’ Response Time
assist = assist[["user_id", "problem_id", "correct", "ms_first_response"]].copy()
assist = assist.rename(columns={
    "user_id":           "Student ID",
    "problem_id":        "Question ID",
    "ms_first_response": "Response Time",
})

# Enforce types
assist["correct"]       = assist["correct"].astype(int)
assist["Response Time"] = assist["Response Time"].astype(float)

# Remove outliers  responses outside [1s, 5min] are not useful signal
before = len(assist)
assist = assist[
    (assist["Response Time"] >= MS_FLOOR) &
    (assist["Response Time"] <= MS_CAP)
]
removed = before - len(assist)
print(f"Removed {removed:,} outlier rows ({removed/before*100:.1f}%) outside [{MS_FLOOR:,}, {MS_CAP:,}] ms")
print(f"Clean shape: {assist.shape}")


Removed 6,226 outlier rows (1.8%) outside [1,000, 300,000] ms
Clean shape: (340634, 4)


In [8]:
assist['Response Time'].describe()

count    340634.000000
mean      33601.888881
std       42330.899537
min        1000.000000
25%        8139.000000
50%       18144.000000
75%       40669.000000
max      299966.000000
Name: Response Time, dtype: float64

# 2. Splitting the data
 - To avoid data leakage we need to split the data before creating the additional rate features

In [9]:
# Split on the RAW DataFrame â€” no features computed yet
train_raw, test_raw = train_test_split(assist, test_size=0.2, random_state=42)

print(f"Train : {len(train_raw):,} rows")
print(f"Test  : {len(test_raw):,}  rows")


Train : 272,507 rows
Test  : 68,127  rows


# 3. Feature engineering

In [10]:
# Compute all aggregates from TRAINING data only
q_rate, s_rate, global_q_mean, global_s_mean = compute_rates_from_train(train_raw)

print(f"Unique questions in train : {len(q_rate):,}")
print(f"Unique students  in train : {len(s_rate):,}")
print(f"Global question correct mean (fallback) : {global_q_mean:.4f}")
print(f"Global student  correct mean (fallback) : {global_s_mean:.4f}")


Unique questions in train : 25,807
Unique students  in train : 4,169
Global question correct mean (fallback) : 0.6416
Global student  correct mean (fallback) : 0.6264


In [11]:
# Apply train-derived rates to BOTH splits
# Test set uses the same rates computed above â€” no test data involved
train_df = apply_rates(train_raw, q_rate, s_rate, global_q_mean, global_s_mean)
test_df  = apply_rates(test_raw,  q_rate, s_rate, global_q_mean, global_s_mean)

# Log-transform target: Response Time is heavily right-skewed
train_df["log_response_time"] = np.log1p(train_df["Response Time"])
test_df["log_response_time"]  = np.log1p(test_df["Response Time"])

print("Difficulty distribution (train set):")
dist = train_df["difficulty"].value_counts()
for lvl in ["easy", "medium", "hard"]:
    count = dist.get(lvl, 0)
    pct   = count / len(train_df) * 100
    print(f"  {lvl:<8}: {count:>7,} rows  ({pct:.1f}%)")


Difficulty distribution (train set):
  easy    : 145,912 rows  (53.5%)
  medium  :  95,800 rows  (35.2%)
  hard    :  30,795 rows  (11.3%)


In [12]:
# Response time stats by difficulty â€” confirms difficulty labels make sense
train_df.groupby("difficulty")["Response Time"] \
        .describe()[["mean", "std", "min", "max"]] \
        .round(0)


,mean,std,min,max
difficulty,,,,
easy,28512.0,37269.0,1000.0,299957.0
hard,37848.0,47734.0,1000.0,299406.0
medium,40049.0,46569.0,1000.0,299966.0


# 4. Model training & comparaison

In [13]:
X_train = train_df[FEATURES].values
y_train = train_df["log_response_time"].values

X_test  = test_df[FEATURES].values
y_test  = test_df["log_response_time"].values

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")


X_train : (272507, 4)
X_test  : (68127, 4)


In [14]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
)

gb = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    min_samples_leaf=20,
    random_state=42,
)

results = []

for name, m in [("Random Forest", rf), ("Gradient Boosting", gb)]:
    print(f"Training {name}...")
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    res   = print_eval(name, y_test, preds)
    res["model"] = m
    results.append(res)
    print()


Training Random Forest...
  [Random Forest]
    RÂ²   (log scale) : 0.1062
    MAE  (ms)        :     23,957   (~24.0s average error)
    RMSE (ms)        :     43,674

Training Gradient Boosting...
  [Gradient Boosting]
    RÂ²   (log scale) : 0.1229
    MAE  (ms)        :     23,770   (~23.8s average error)
    RMSE (ms)        :     43,460



In [15]:
# Select best model by RÂ²
best  = max(results, key=lambda r: r["r2"])
model = best["model"]
print(f"Best model: {best['name']}  (RÂ² = {best['r2']:.4f})")


Best model: Gradient Boosting  (RÂ² = 0.1229)


In [16]:
# Feature importances
importances = pd.Series(model.feature_importances_, index=FEATURES) \
                .sort_values(ascending=False)

print(f"Feature importances ({best['name']}):")
for feat, imp in importances.items():
    print(f"  {feat:<28}: {imp:.4f}")


Feature importances (Gradient Boosting):
  student_correct_rate        : 0.5582
  question_correct_rate       : 0.4008
  correct                     : 0.0410
  difficulty_encoded          : 0.0001


In [17]:
# Comparison summary table
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k != "model"}
    for r in results
]).set_index("name")

summary[["r2", "mae", "rmse"]].round({"r2": 4, "mae": 0, "rmse": 0})


,r2,mae,rmse
name,,,
Random Forest,0.1062,23957.0,43674.0
Gradient Boosting,0.1229,23770.0,43460.0


In [18]:
# Save best model + the train-derived rates needed for 
import pickle
with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump({
        "model":          model,
        "model_name":     best["name"],
        "features":       FEATURES,
        "q_rate":         q_rate,          # train-derived, used at predict time
        "s_rate":         s_rate,          # train-derived, used at predict time
        "global_q_mean":  global_q_mean,   # fallback for unseen questions
        "global_s_mean":  global_s_mean,   # fallback for unseen students
        "easy_threshold": EASY_THRESHOLD,
        "hard_threshold": HARD_THRESHOLD,
        "ms_cap":         MS_CAP,
        "ms_floor":       MS_FLOOR,
    }, f)

print(f"Model saved â†’ {MODEL_SAVE_PATH}")


Model saved â†’ ../models/response_time_model.pkl


# 5. Using the model to fill the real dataset

In [19]:
real_df = pd.read_csv(REAL_DATASET_PATH)
print(f"Shape: {real_df.shape}")
real_df.head()


Shape: (17340, 5)


,Student ID,Question ID,topic,correct,Response Time
0,AQklHbi5bt8l,1,NaN,incorrect,NaN
1,AQklHbi5bt8l,2,NaN,correct,NaN
2,AQklHbi5bt8l,3,NaN,correct,NaN
3,AQklHbi5bt8l,4,NaN,correct,NaN
4,AQklHbi5bt8l,5,NaN,incorrect,NaN


- Encoding the correct feature

In [20]:
# Convert correct: "correct" â†’ 1, "incorrect" â†’ 0
real_df["correct"] = real_df["correct"].map(
    {"correct": 1, "incorrect": 0}
).astype(int)

print("Correct value counts:")
print(real_df["correct"].value_counts().to_string())


Correct value counts:
correct
1    13134
0     4206


- Calculating the rates

In [21]:

real_q_rate = real_df.groupby("Question ID")["correct"].mean()
real_s_rate = real_df.groupby("Student ID")["correct"].mean()

# Apply our own rates  global means are fallback only and they won't be triggered in this case
real_df = apply_rates(real_df, real_q_rate, real_s_rate, global_q_mean, global_s_mean)

print("Difficulty distribution in the dataset:")
dist = real_df["difficulty"].value_counts()
for lvl in ["easy", "medium", "hard"]:
    count = dist.get(lvl, 0)
    pct   = count / len(real_df) * 100 if len(real_df) > 0 else 0
    print(f"  {lvl:<8}: {count:>6} rows  ({pct:.1f}%)")

print()
print("Per-question correct rates (top 10):")
rate_df = real_q_rate.reset_index()
rate_df.columns = ["Question ID", "correct_rate"]
rate_df["difficulty"] = rate_df["correct_rate"].apply(assign_difficulty)
print(rate_df.sort_values("correct_rate", ascending=False).head(10).to_string(index=False))


Difficulty distribution in the dataset:
  easy    :  12583 rows  (72.6%)
  medium  :   2546 rows  (14.7%)
  hard    :   2211 rows  (12.8%)

Per-question correct rates (top 10):
 Question ID  correct_rate difficulty
          10      1.000000       easy
           8      1.000000       easy
          11      1.000000       easy
          94      1.000000       easy
          93      1.000000       easy
          12      0.991453       easy
          61      0.991453       easy
          29      0.991453       easy
           4      0.991453       easy
          65      0.991453       easy


### Predicting the response time using the saved model

In [22]:
#load the model
with open(MODEL_SAVE_PATH, "rb") as f:
    saved = pickle.load(f)
    model = saved["model"]
    print(f"Loaded model: {saved['model_name']}  (features: {saved['features']})")
    




Loaded model: Gradient Boosting  (features: ['correct', 'difficulty_encoded', 'student_correct_rate', 'question_correct_rate'])


In [23]:

X_real= real_df[FEATURES].values
log_preds = model.predict(X_real)
ms_preds  = np.clip(np.expm1(log_preds).astype(int), MS_FLOOR, MS_CAP)

real_df["Response Time"] = ms_preds

print(f"Filled Response Time for {len(real_df)} rows")
print()
print("Predicted stats by difficulty:")
real_df.groupby("difficulty")["Response Time"] \
       .describe()[["mean", "std", "min", "max"]] \
       .round(0)


Filled Response Time for 17340 rows

Predicted stats by difficulty:


,mean,std,min,max
difficulty,,,,
easy,15853.0,4244.0,6361.0,41012.0
hard,25632.0,4402.0,9325.0,46882.0
medium,28205.0,4447.0,15661.0,45204.0


# 6. Saving the dataset to clean

In [24]:
output_df = real_df[[
    "Student ID", "Question ID", "topic",
    "difficulty", "correct", "Response Time"
]].copy()

output_df.to_csv(OUTPUT_PATH, index=False)

print("Null check:")
print(output_df.isnull().sum().to_string())
print()
print(f"Saved to {OUTPUT_PATH}")


Null check:
Student ID           0
Question ID          0
topic            17340
difficulty           0
correct              0
Response Time        0

Saved to ../Dataset/clean/interactions_clean.csv
